# 3-lens System using MeepSAT

In this Tutorial, we are going to simulate a complex 3 lens system with all the possible components currently availabe in MeepSATs functionality. We will be simulating the system in low-frequency (LF) range (90, 120, 150 GHz).

We will be using no AR Coatings in these simulations since it requires a minimum of resolution 19 pixels per mm (which isn't feasible to run in your local device). But later, we will give a tutorial on how to do that on a HPC facility.

The general workflow is the same that we have shown in the previous examples (https://github.com/aa16oaslak/MeepSAT/blob/main/examples/01_simple_single_lens_ARC.ipynb)

In [2]:
# Importing various Python libraries and MeepSAT modules
import sys
import os
import site
from pathlib import Path
import meep as mp
import numpy as np
import h5py
import matplotlib.pyplot as plt
import time
import json
import math

# Importing the MEEPSAT librarires
import meepsat.simulator as sim
import meepsat.meep_geometry as comp_meep
import meepsat.permittivity_components as comp_eps
import meepsat.stepfunctions as stepfunctions
import meepsat.json_to_script as json_to_script
import meepsat.field_analysis as mpsat_analysis
import meepsat.helpers as mpsat_helpers

# JSON file path representing mainly the different optical components parameters
json_file_path = 'auxilary_data/04_3lens_system_noARC/3lens_system_noARC.json'
data = mpsat_helpers.read_json(json_file_path)

# Some universal constants
c_mm_s = 299792458.0 * 1000.0  # Speed of light in mm/s (m/s -> mm/s)
# Frequency of the simulation
freq = 150.0  # Frequency in GHz
a = 1  # 1 meep unit = 1 mm  
wvl = c_mm_s / (freq * 1e9)  # Wavelength in mm
freq_meep = 1.0 / (wvl * a)
print("freq (meep units):", freq_meep)

# Edit the freq in the JSON file 
data["sources"]["source1"]["frequecy"] = freq_meep

# Savepath: For storing the output generated during the simulation
savepath = f'auxilary_data/04_3lens_system_noARC/TRV/output_files/{freq}GHz'
os.makedirs(savepath, exist_ok=True)
data["output"]["savepath"]["path"] = savepath

# Initialising MEEPSAT Simulation
cell_X, cell_Y, cell_Z = data["simulation"]['primary_params']['cell_size']['x'], data["simulation"]['primary_params']['cell_size']['y'], data["simulation"]['primary_params']['cell_size']['z'] # Cell Size without considering the PML thickness and its factor


# Initialize the simulation with the different parameters
mpsat_sim = sim.sim_init(sim_name= str(data["simulation"]["name"]),
                        cell_size= [cell_X, cell_Y, cell_Z], # [sx, sy, sz] in mm
                        smallest_freq= data["simulation"]['primary_params']['smallest_freq'], 
                        resolution= data["simulation"]['primary_params']['resolution'],
                        boundary_layer_type= data['boundary_layers']['boundary']['type'],
                        boundary_layer_size= data['boundary_layers']['boundary']['size'],
                        factor_dpml= data['boundary_layers']['boundary']['factor_dpml'])

# Checking resolution and PML thickness 
# This function will automatically check all the length scales and wavelength scales
# data, mpsat_sim = sim.check_resolution_and_pml(
#     data=data, 
#     mpsat_sim=mpsat_sim,
#     smallest_freq=data["simulation"]['primary_params']['smallest_freq'],
#     highest_n=data["lenses"]["lens1"]["n_refr"]
# )

# Print the simulation parameters
print("\nMEEPSAT SIMULATION PARAMETERS:")
mpsat_sim.print_simulation_parameters()


# Boundary Layers
x_left_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.Low)
x_right_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.High)
y_down_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.Low)
y_up_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.High)

custom_boundary_layers = [x_left_boundary, x_right_boundary, y_down_boundary, y_up_boundary]

# Creating the epsilon map for the simulation cell
size_x, size_y, size_z = mpsat_sim.cell_size[0], mpsat_sim.cell_size[1], mpsat_sim.cell_size[2]
res = int(mpsat_sim.resolution)  # Ensure resolution is an integer
# Create the epsilon map: total size of the simulation cell in all the axis multiplied by the resolution + 1
epsilon_map = np.ones((int((size_x)*res+1), 
                       int((size_y)*res+1)), dtype = float)

# Adding Source
source_list = []
exec(json_to_script.source_script(data))

# Adding lens (if given)
exec(json_to_script.add_lens(data))

# Adding box slabs (if given)
exec(json_to_script.add_slab(data))

# Adding aperture (if given)
exec(json_to_script.add_aperture(data))

freq (meep units): 0.5003461427972281

MEEPSAT SIMULATION PARAMETERS:
Simulation name: 3lens_system_noARC
Cell size: [1630, 834, 0]
Frequency: 0.25
Wavelength: 4.0
Resolution: 3
Boundary layer type: PML
Boundary layer size: 2
Factor for PML boundary layer thickness: 2
Angle of the source:3.141592653589793 rad = 180 degrees
Additional arguments for the ContinuousSource:  {'start_time': 0, 'end_time': 1e+20}
Additional arguments for GaussianBeamSource:  {'beam_x0': Vector3<0.0, 0.0, 0.0>, 'beam_E0': Vector3<0.0, 0.0, 1.0>}
Gaussian beam source assembled!
Saving epsilon map to HDF5 file...
[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]] added to the list of components created using the epsilon functions!
[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]] added to the list of comp

Now let's plot the epsilon map and visualise the results

In [3]:
# DEFINING THE SIMULATION OBJECT
# Mirror symmetry along y direction (x=0 plane)
symmetries = [mp.Mirror(mp.Y, phase=+1)] 

simulation = mp.Simulation(
    cell_size=mpsat_sim.cell,
    sources=source_list,
    resolution=mpsat_sim.resolution,
    boundary_layers=custom_boundary_layers,
    geometry=mpsat_sim.meep_geometry,
    epsilon_input_file = data["output"]["savepath"]["path"] + data["output"]["epsilon_h5_file"]["filename"] +"_epsilon_map" + ".h5",
    symmetries = symmetries,
    force_complex_fields= True)

simulation.use_output_directory(savepath)
# ---------------------------------------------

# Run simulation briefly to store the epsilon map
sim.plot_and_save_epsilon(
    simulation=simulation,
    savepath=savepath,
    filename_prefix="geometry_plot",
    epsilon_data_name="epsilon",
    size_x=size_x,
    size_y=size_y,
    vmin=0.5,
    vmax=3,
    cmap='viridis',
    figsize=(8, 4),
    dpi=300
)

-----------
Initializing structure...
read in 4891x2503x1 epsilon-input-file "auxilary_data/04_3lens_system_noARC/TRV/output_files/150.0GHz3lens_system_noARC_epsilon_map.h5"
Halving computational cell along direction y
time for choose_chunkdivision = 0.001786 s
read in 4891x2503x1 epsilon-input-file "auxilary_data/04_3lens_system_noARC/TRV/output_files/150.0GHz3lens_system_noARC_epsilon_map.h5"
Working in 2D dimensions.
Computational cell is 1630 x 834 x 0 with resolution 3
     block, center = (1056,263.5,0)
          size (2,307,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (-1e+20,-1e+20,-1e+20)
     block, center = (1056,-263.5,0)
          size (2,307,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (-1e+20,-1e+20,-1e+20)
     block, center = (1050.5,270.848,0)
          size (1,292.304,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (-1e+20,-1e+2

FloatProgress(value=0.0, description='0% done ', max=0.0)

run 0 finished at t = 0.0 (0 timesteps)
Epsilon plot saved to: auxilary_data/04_3lens_system_noARC/TRV/output_files/150.0GHz/geometry_plot.png
Epsilon data saved to: auxilary_data/04_3lens_system_noARC/TRV/output_files/150.0GHz/geometry_plot.h5
